# Figure 2 — LinkD-Select drug selectivity

Proteome-wide selectivity, oncogene enrichment, known-DTI recovery, radar profiles, and docking validation.

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `drug_selectivity_metrics.csv`, `target_binding_stats.csv`, `onco_genes.csv`
- `opentarget_known_drug_pair.csv`, `radar_egfr_jak1_fig2e.csv`
- `docking_scores_fig2fg.csv` (extracted subset)


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panel a — Affinity vs entropy scatter (14,981 drugs)

In [ ]:

sel = io.read_selectivity()
# Prefer normalized columns when present
xcol = "aff_n" if "aff_n" in sel.columns else "Selectivity_Score"
ycol = "entropy_sel_n" if "entropy_sel_n" in sel.columns else "entropy_norm"
fig, ax = plt.subplots(figsize=(4.5, 3.8))
sc = ax.scatter(sel[xcol], sel[ycol], c=sel["Selectivity_Score"], s=3, cmap="viridis", alpha=0.35, linewidths=0)
ax.set_xlabel(xcol)
ax.set_ylabel(ycol)
ax.set_title("Fig 2a — affinity vs entropy (colored by Selectivity_Score)")
fig.colorbar(sc, ax=ax, label="Selectivity_Score", fraction=0.046)
fig.tight_layout()
out = style.save_panel(fig, "fig2_a_scatter", sel[[xcol, ycol, "Selectivity_Score", "Drug Chembl ID"]].dropna())
plt.show()
print(out)


## Panel b — Median selectivity by cancer gene role (lollipop)

In [ ]:

def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(q, 0, 1)
    return out

tbs = io.read_target_stats()
onco = io.read_onco_genes()
# Merge role
if "Role" not in tbs.columns or tbs["Role"].isna().all():
    tbs = tbs.merge(onco, left_on="Gene", right_on="Gene", how="left", suffixes=("", "_onco"))
    if "Role_onco" in tbs.columns:
        tbs["Role"] = tbs["Role"].fillna(tbs["Role_onco"])
score = "Avg_Selectivity_Score" if "Avg_Selectivity_Score" in tbs.columns else "Selectivity_Score"
bg = tbs[score].dropna()
# Keep cancer-relevant annotated genes
sub = tbs[tbs["Role"].isin(["oncogene", "TSG", "both", "Oncogene", "Tumor Suppressor", "Dual"])].copy()
# Normalize role labels
role_map = {"Oncogene": "oncogene", "Tumor Suppressor": "TSG", "Dual": "both", "both": "both", "oncogene": "oncogene", "TSG": "TSG"}
sub["Role"] = sub["Role"].map(lambda x: role_map.get(x, x))
pvals = []
for _, r in sub.iterrows():
    p = (bg >= r[score]).mean() if pd.notna(r[score]) else 1.0
    pvals.append(max(p, 1e-12))
sub["p_raw"] = pvals
sub["p_fdr"] = bh_fdr(sub["p_raw"])
sub = sub.sort_values(score, ascending=False).head(60)

colors = {"oncogene": style.PALETTE["oncogene"], "TSG": style.PALETTE["tsg"], "both": style.PALETTE["dual"]}
fig, ax = plt.subplots(figsize=(7.5, 3.2))
x = np.arange(len(sub))
for role, g in sub.groupby("Role"):
    idx = [i for i, rr in enumerate(sub["Role"]) if rr == role]
    ax.vlines(idx, 0, sub.iloc[idx][score], color=colors.get(role, "gray"), linewidth=0.8)
    sig = sub.iloc[idx]["p_fdr"] < 0.05
    ax.scatter(np.array(idx)[sig], sub.iloc[idx][score][sig], color=colors.get(role, "gray"), s=18, label=f"{role} FDR<0.05")
    ax.scatter(np.array(idx)[~sig], sub.iloc[idx][score][~sig], facecolors="none", edgecolors=colors.get(role, "gray"), s=18, label=f"{role} n.s.")
ax.set_xticks(x)
ax.set_xticklabels(sub["Gene"].fillna(sub.get("Target", "")), rotation=90, fontsize=5)
ax.set_ylabel("Median / Avg Selectivity")
ax.set_title("Fig 2b — selectivity by gene role")
ax.legend(frameon=False, fontsize=5, ncol=3)
fig.tight_layout()
out = style.save_panel(fig, "fig2_b_lollipop", sub[["Gene", "Role", score, "p_fdr"]])
plt.show()
print(out)


## Panels c–d — Known DTI recovery at top 5% / 10%

In [ ]:

# Approximate recovery using target_binding_stats + known pairs when full pair matrix unavailable.
# Prefer CRISPR file which already joins known status with selectivity/affinity ranks when available.
cr = io.read_crispr()
# Per-drug ranking by Selectivity_Score / aff_n / combined
need = {"Drug Chembl ID", "Selectivity_Score", "is_known"}
if need.issubset(cr.columns):
    d = cr.dropna(subset=["Selectivity_Score"]).copy()
    d["is_known"] = d["is_known"].astype(str).str.lower().isin(["1", "true", "yes", "known"])
    aff = "aff_n" if "aff_n" in d.columns else "Target_Affinity"
    d["combined"] = d["Selectivity_Score"].rank(pct=True) + d[aff].rank(pct=True)
    rows = []
    for thr, lab in [(0.05, "top5"), (0.10, "top10")]:
        for score_name, col in [("affinity", aff), ("selectivity", "Selectivity_Score"), ("combined", "combined")]:
            # within each drug, take top thr fraction
            def top_frac(g):
                k = max(1, int(np.ceil(thr * len(g))))
                return g.nlargest(k, col)
            tops = d.groupby("Drug Chembl ID", group_keys=False).apply(top_frac, include_groups=False) if hasattr(pd.core.groupby.DataFrameGroupBy.apply, "__call__") else d.groupby("Drug Chembl ID", group_keys=False).apply(top_frac)
            try:
                tops = d.groupby("Drug Chembl ID", group_keys=False).apply(lambda g: top_frac(g))
            except TypeError:
                tops = d.groupby("Drug Chembl ID", group_keys=False).apply(top_frac)
            known_all = d[d["is_known"]]
            known_rec = tops[tops["is_known"]]
            # recovery = fraction of known pairs recovered
            denom = known_all.drop_duplicates(["Drug Chembl ID", "Gene"]).shape[0]
            num = known_rec.drop_duplicates(["Drug Chembl ID", "Gene"]).shape[0]
            rows.append({"threshold": lab, "strategy": score_name, "recovered": num, "known_total": denom, "frac": num / denom if denom else np.nan})
    rec = pd.DataFrame(rows)
else:
    rec = pd.DataFrame({"threshold": [], "strategy": [], "frac": []})
    print("CRISPR file missing expected columns; writing empty recovery table")

fig, axes = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
for ax, thr in zip(axes, ["top5", "top10"]):
    sub = rec[rec["threshold"] == thr]
    if sub.empty:
        ax.text(0.5, 0.5, "n/a", ha="center")
    else:
        ax.bar(sub["strategy"], sub["frac"], color=[style.PALETTE["affinity"], style.PALETTE["selectivity"], style.PALETTE["combined"]])
        ax.axhline(float(thr.replace("top", ""))/100 if False else 0.05 if thr=="top5" else 0.10, ls="--", color="gray", lw=0.8, label="random")
    ax.set_title(f"Fig 2{'c' if thr=='top5' else 'd'} — {thr}")
    ax.set_ylabel("Fraction known DTIs recovered")
fig.tight_layout()
out = style.save_panel(fig, "fig2_cd_recovery", rec)
plt.show()
print(out)
display(rec)


## Panel e — EGFR / JAK1 radar profiles

In [ ]:

radar = io.read_radar()
# Normalize metrics 0-1 per column for radar
metrics = [c for c in ["Selectivity_Score", "entropy_sel_n", "gap_local_n", "sr_local_n", "aff_local_n"] if c in radar.columns]
if not metrics:
    metrics = [c for c in radar.columns if radar[c].dtype != object][:5]
# Identify target
tcol = "Target"
radar["_gene"] = radar[tcol].astype(str).str.upper().str.extract(r"(EGFR|JAK1)")[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), subplot_kw=dict(polar=True))
for ax, gene in zip(axes, ["EGFR", "JAK1"]):
    g = radar[radar["_gene"] == gene].copy()
    if g.empty:
        ax.set_title(gene + " (no data)")
        continue
    g = g.nlargest(5, "Selectivity_Score")
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]
    for _, row in g.iterrows():
        vals = []
        for m in metrics:
            col = g[m]
            v = (row[m] - col.min()) / (col.max() - col.min() + 1e-12)
            vals.append(v)
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=1, label=str(row.get("Drug", ""))[:12])
        ax.fill(angles, vals, alpha=0.05)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=6)
    ax.set_title(gene)
    ax.legend(fontsize=5, loc="upper right", bbox_to_anchor=(1.35, 1.1))
fig.suptitle("Fig 2e — radar profiles")
fig.tight_layout()
out = style.save_panel(fig, "fig2_e_radar", radar)
plt.show()
print(out)


## Panel f — Docking score on-target vs off-target by role

In [ ]:

dock = io.read_docking()
onco = io.read_onco_genes()
dock = dock.merge(onco, left_on="Gene", right_on="Gene", how="left")
dock["on_target"] = dock["Type"].astype(str).str.lower().eq("known")
dock = dock.dropna(subset=["Docking Score"])
# Role cleanup
dock["Role"] = dock["Role"].fillna("unknown")
fig, ax = plt.subplots(figsize=(5.5, 3.5))
roles = [r for r in ["oncogene", "TSG", "both", "Oncogene", "Tumor Suppressor"] if r in set(dock["Role"])]
if not roles:
    roles = sorted(dock["Role"].value_counts().head(3).index)
data_rows = []
positions = []
pos = 0
labels = []
for role in roles:
    for ot, name in [(True, "on"), (False, "off")]:
        vals = dock[(dock["Role"] == role) & (dock["on_target"] == ot)]["Docking Score"].values
        if len(vals) == 0:
            continue
        ax.violinplot([vals], positions=[pos], showmeans=False, showmedians=True, widths=0.8)
        data_rows.append({"Role": role, "class": name, "n": len(vals), "median": np.median(vals)})
        labels.append(f"{role}\n{name}")
        positions.append(pos)
        pos += 1
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=6)
ax.set_ylabel("Docking Score (kcal/mol)")
ax.set_title("Fig 2f — docking on vs off target")
fig.tight_layout()
out = style.save_panel(fig, "fig2_f_docking_raincloud", pd.DataFrame(data_rows))
plt.show()
print(out)


## Panel g — Cumulative recovery vs docking cutoff

In [ ]:

dock = io.read_docking()
known = dock[dock["Type"].astype(str).str.lower().eq("known")].dropna(subset=["Docking Score"])
cutoffs = np.arange(-12, -1.5, 0.5)
fracs = []
for c in cutoffs:
    fracs.append((known["Docking Score"] <= c).mean())
rec = pd.DataFrame({"cutoff": cutoffs, "frac_recovered": fracs})
fig, ax = plt.subplots(figsize=(4.2, 3.2))
ax.plot(rec["cutoff"], rec["frac_recovered"], color=style.PALETTE["linkd"], lw=1.5)
for c in [-8, -7, -6]:
    y = (known["Docking Score"] <= c).mean()
    ax.axvline(c, color="gray", ls=":", lw=0.7)
    ax.text(c, y, f"{y*100:.1f}%", fontsize=6)
ax.set_xlabel("Docking score cutoff (kcal/mol)")
ax.set_ylabel("Fraction known pairs recovered")
ax.set_title("Fig 2g — cumulative docking recovery")
fig.tight_layout()
out = style.save_panel(fig, "fig2_g_docking_recovery", rec)
plt.show()
print(out)
